In [1]:
import torch
import torch.nn as nn

In [2]:
class SeqToVecLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, num_layers):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers=num_layers, batch_first=True)
        self.output = nn.Linear(hidden_size, output_size)

    def forward(self, X):
        outputs, (h_n, c_n) = self.lstm(X)
        return self.output(outputs[:, -1])

In [3]:
class SeqToSeqLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, num_layers):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers=num_layers, batch_first=True)
        self.output = nn.Linear(hidden_size, output_size)

    def forward(self, X):
        outputs, (h_n, c_n) = self.lstm(X)
        return self.output(outputs)

In [4]:
class VecToSeqLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, num_layers, seq_len):
        super().__init__()
        self.seq_len = seq_len
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers=num_layers, batch_first=True)
        self.output = nn.Linear(hidden_size, output_size)

    def forward(self, x_vec):
        X_repeated = x_vec.unsqueeze(1).repeat(1, self.seq_len, 1)
        outputs, (h_n, c_n) = self.lstm(X_repeated)
        return self.output(outputs)

In [ ]:
class EncoderDecoderLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, num_layers, seq_len):
        super().__init__()
        self.seq_len = seq_len
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.encoder = nn.LSTM(input_size, hidden_size, num_layers=num_layers, batch_first=True)
        self.decoder = nn.LSTM(hidden_size, hidden_size, num_layers=num_layers, batch_first=True)
        self.output = nn.Linear(hidden_size, output_size)

    def forward(self, X):
        batch_size = X.shape[0]
        _, (h_n, c_n) = self.encoder(X)
        decoder_input = torch.zeros(batch_size, self.seq_len, self.hidden_size)
        outputs, _ = self.decoder(decoder_input, (h_n, c_n))
        return self.output(outputs)